In [2]:
#Importing the Libraries

!pip install simpletransformers
!pip install transformers-interpret

import pandas
import numpy
from ast import literal_eval
from sklearn.metrics import f1_score, classification_report
from simpletransformers.classification import MultiLabelClassificationModel
from transformers_interpret import SequenceClassificationExplainer

In [3]:
trainPath = "../input/raw-data/GoE - RawTrain - NoD.csv"
testPath = "../input/raw-data/GoE - RawTest - NoD.csv"

emoList = ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 
'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism',
'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']

ekmanDictMo = {
    "anger": ["anger", "annoyance", "disapproval"],
    "disgust": ["disgust"],
    "fear": ["fear", "nervousness"],
    "joy": ["joy", "amusement", "approval", "excitement", "gratitude",  "love", "optimism", "relief", "pride", "admiration", "desire", "caring"],
    "sadness": ["sadness", "disappointment", "embarrassment", "grief",  "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral": ["neutral"]
}

ekmanList = list(ekmanDictMo.keys())

In [4]:
train_df = pandas.read_csv(trainPath)

trainCol = []
for label in train_df['label']:
    tempL = [0] * len(ekmanList)
    for indiLab in literal_eval(label):
        for key, value in ekmanDictMo.items():
            if indiLab in value:
                tempL[ekmanList.index(key)] = 1
    trainCol.append(tempL)
train_df['label_desc'] = train_df['label']
train_df['label'] = trainCol
train_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

train_df.head()

In [5]:
test_df = pandas.read_csv(testPath)

testCol = []
for label in test_df['label']:
    tempL = [0] * len(ekmanList)
    for indiLab in literal_eval(label):
        for key, value in ekmanDictMo.items():
            if indiLab in value:
                tempL[ekmanList.index(key)] = 1
    testCol.append(tempL)
test_df['label_desc'] = test_df['label']
test_df['label'] = testCol
test_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

test_df.head()

In [6]:
# Training Roberta Transformer on GoEmotions as MultiLabel Classification

model = MultiLabelClassificationModel('roberta', 'roberta-base', num_labels=len(ekmanList), 
                                      args={'train_batch_size':4, 'fp16': False, 'overwrite_output_dir': True,                                         
                                            'gradient_accumulation_steps':8, 'learning_rate': 3e-5,
                                            'num_train_epochs': 1, "reprocess_input_data": True, 
                                            'max_seq_length': 128})

model.train_model(train_df)
result, model_outputs, wrong_predictions = model.eval_model(test_df)
predictions, raw_outputs = model.predict(list(test_df['text']))

print('-'*100)

print(f"Accuracy & Loss - {result}")
print(f"F1-Micro Score - {f1_score(testCol, predictions, average='micro')}")
print(f"F1-Macro Score - {f1_score(testCol, predictions, average='macro')}")